In [1]:
  !pip install langchain langchain-community openai feedparser

  Using cached langchain-1.2.15-py3-none-any.whl.metadata (5.8 kB)
  Using cached langchain_community-0.4.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached openai-2.31.0-py3-none-any.whl.metadata (31 kB)
  Using cached langgraph-1.1.6-py3-none-any.whl.metadata (8.0 kB)
  Using cached pydantic-2.13.0-py3-none-any.whl.metadata (107 kB)
  Using cached jsonpatch-1.33-py2.py3-none-any.whl.metadata (3.0 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached uuid_utils-0.14.1-cp39-abi3-macosx_10_12_x86_64.macosx_11_0_arm64.macosx_10_12_universal2.whl.metadata (4.8 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached langgraph_checkpoint-4.0.1-py3-none-any.whl.metadata (4.9 kB)
  Using cached langgraph_prebuilt-1.0.9-py3-none-any.whl.metadata (5.2 kB)
  Using cached langgraph_s

In [6]:
!pip install langchain-openai

In [2]:
import os

os.environ["OPENAI_API_KEY"] = "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"
os.environ["OPENAI_API_BASE"] = "https://openrouter.ai/api/v1"

In [8]:
import os
from langchain_openai import ChatOpenAI

#os.environ["OPENAI_API_KEY"] = "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"

llm = ChatOpenAI(model="openai/gpt-oss-120b:free",temperature=0.7)

In [9]:
llm

ChatOpenAI(output_version=None, client=<openai.resources.chat.completions.completions.Completions object at 0x10f57ca50>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10f57ce10>, root_client=<openai.OpenAI object at 0x10f57c7d0>, root_async_client=<openai.AsyncOpenAI object at 0x10f57cb90>, model_name='openai/gpt-oss-120b:free', temperature=0.7, model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://openrouter.ai/api/v1', openai_proxy=None)

In [15]:
import feedparser
import requests

rss_url = "https://techcrunch.com/tag/artificial-intelligence/feed/"
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
}

response = requests.get(rss_url, headers=headers, timeout=20)
response.raise_for_status()
feed = feedparser.parse(response.content)

articles = [entry.get("title", "").strip() for entry in feed.entries if entry.get("title")]
articles = articles[:10]

if not articles:
    raise ValueError("No headlines fetched from RSS feed. Try a different feed URL.")

print(f"Fetched {len(articles)} headlines")
articles

Fetched 10 headlines


['Converge Bio raises $25M, backed by Bessemer and execs from Meta, OpenAI, Wiz',
 'Meta bought 1 GW of solar this week',
 'How one AI startup is helping rice farmers battle climate change',
 'Harvard dropouts to launch ‘always on’ AI smart glasses that listen and record every conversation',
 'Meta to add 100MW of solar power from US gear',
 'Perplexity accused of scraping websites that explicitly blocked AI scraping',
 'Obvio’s stop sign cameras use AI to root out unsafe drivers',
 'Breakneck data center growth challenges Microsoft’s sustainability goals',
 'Gridcare thinks more than 100 GW of data center capacity is hiding in the grid',
 'Meta adds another 650 MW of solar power to its AI push']

In [16]:
from langchain_core.prompts import PromptTemplate

research_prompt = PromptTemplate.from_template(
    """
You are a research agent.

Analyze the following AI news headlines:
{articles}

Return:
- key topics
- short summaries
- important keywords

Output in structured format.
"""
)

formatted_articles = "\n".join(f"- {title}" for title in articles)
research_chain = research_prompt | llm
research_output = research_chain.invoke({"articles": formatted_articles})
print(research_output.content)

**AI‑News Headlines – Quick Analysis**

| # | Headline | Key Topics | Short Summary (≤ 2 sentences) | Important Keywords |
|---|----------|------------|--------------------------------|--------------------|
| 1 | **Converge Bio raises $25M, backed by Bessemer and execs from Meta, OpenAI, Wiz** | • AI‑driven biotech financing  <br>• Venture capital involvement  <br>• Cross‑industry talent (Meta, OpenAI, Wiz) | Converge Bio, a biotech startup that uses generative‑AI for protein/antibody design, closed a $25 million Series A round. The round was led by Bessemer Venture Partners and included individual investors from Meta, OpenAI and cybersecurity firm Wiz. | Converge Bio, $25 M, Bessemer, Meta, OpenAI, Wiz, generative AI, biotech, protein design |
| 2 | **Meta bought 1 GW of solar this week** | • Corporate renewable‑energy procurement  <br>• AI‑heavy data‑center power needs  <br>• ESG strategy | Meta announced the purchase of 1 gigawatt of solar power contracts to offset the electricity c

In [17]:
trend_prompt = PromptTemplate(
    input_variables=["data"],
    template="""
You are a trend detection agent.

Based on this research:
{data}

Identify:
- top trending topics
- why they are trending

Return only the top 3 topics.
"""
)

trend_chain = LLMChain(llm=llm, prompt=trend_prompt)

trends = trend_chain.run(data=research_output)
print(trends)

**1. Renewable‑energy‑powered AI compute**  
*Why it’s trending:*  Meta’s repeated multi‑hundred‑megawatt solar purchases (1 GW + 650 MW + 100 MW) and the broader strain on grids from exploding data‑center capacity (Microsoft, Gridcare) highlight an industry‑wide push to match the massive power demand of AI models with carbon‑free electricity.  Investors, regulators, and corporate ESG goals are all pressuring the sector to decarbonise its compute infrastructure.

**2. AI‑enabled vertical‑specific solutions**  
*Why it’s trending:*  Start‑ups are embedding generative and computer‑vision AI into niche domains—biotech protein design (Converge Bio), climate‑smart rice farming, AI‑enhanced traffic‑camera enforcement, and “always‑on” smart glasses.  The rapid maturation of foundation models and specialized data pipelines makes AI a viable competitive advantage in sectors that historically lagged behind mainstream tech adoption.

**3. Legal, ethical, and privacy scrutiny of AI data practices*

In [18]:
selection_prompt = PromptTemplate(
    input_variables=["trends"],
    template="""
You are a content strategist.

From these topics:
{trends}

Select:
- ONLY 1 topic worth posting
- based on impact, novelty, and audience interest

Explain why.
"""
)

selection_chain = LLMChain(llm=llm, prompt=selection_prompt)

selected_topic = selection_chain.run(trends=trends)
print(selected_topic)

**Chosen topic:** **Legal, ethical, and privacy scrutiny of AI data practices**  

### Why this beats the other two for a high‑impact, hot‑take post

| Criterion | Renewable‑energy‑powered AI compute | AI‑enabled vertical‑specific solutions | Legal/ethical/privacy scrutiny of AI data |
|-----------|-----------------------------------|----------------------------------------|-------------------------------------------|
| **Impact on the industry** | Important for cost & ESG, but largely a *supply‑chain* optimisation problem that will unfold over years. | Drives niche revenue streams, yet each vertical is still a relatively small slice of total AI spend. | Directly threatens the **core business model** of virtually every AI firm—if data‑use rules tighten, models can’t be trained, deployed, or monetized. |
| **Novelty** | The trend is a logical extension of existing green‑tech initiatives; the narrative is already familiar (solar farms, carbon‑offsets). | Vertical‑specific use‑cases are e

In [19]:
content_prompt = PromptTemplate(
    input_variables=["topic"],
    template="""
You are a Twitter content expert.

Create a viral Twitter post on:
{topic}

Structure:
- Strong hook
- Clear explanation
- Insight
- CTA

Make it engaging and human-like.
"""
)

content_chain = LLMChain(llm=llm, prompt=content_prompt)

post = content_chain.run(topic=selected_topic)
print(post)

**🧵 The data‑pipeline that fuels every AI model is under siege. Here’s why the next wave of lawsuits & privacy bills could **break** the AI boom – and what you can do TODAY to stay afloat. 👇**  

1️⃣ **The trigger:**  
- **Perplexity AI** was hit with a $30 M class‑action for scraping copyrighted web content.  
- **Clearview AI** lost a $650 M judgment for harvesting facial‑images without consent.  
- The **EU AI Act** draft now bans “non‑consensual mass‑scraping” for training.  

🛑 *Bottom line:* If you’re training on public‑web data, you could be a defendant tomorrow.

---

2️⃣ **Why this matters to EVERY AI product**  
🔹 **LLMs, recommendation engines, vision models** – all start with terabytes of scraped data.  
🔹 **Regulators are moving fast:** 12 U.S. states have introduced “AI data‑privacy” bills in the last 6 months.  
🔹 **Fines & shutdowns** are no longer hypothetical – think GDPR‑style penalties: up to 6 % of global revenue.

---

3️⃣ **Three quick risk signals you can spot r

In [20]:
hook_prompt = PromptTemplate(
    input_variables=["post"],
    template="""
Improve ONLY the first line (hook) of this post:

{post}

Make it more engaging and scroll-stopping.
"""
)

hook_chain = LLMChain(llm=llm, prompt=hook_prompt)

optimized_post = hook_chain.run(post=post)
print(optimized_post)

**🚨 Your AI model’s secret weapon is about to become its biggest liability – a looming wave of lawsuits and data‑privacy bans could cripple the whole industry, and you’ve only got 24 hours to shield it.**


In [23]:
!pip install streamlit

  Using cached streamlit-1.56.0-py3-none-any.whl.metadata (9.8 kB)
  Using cached altair-6.0.0-py3-none-any.whl.metadata (11 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached cachetools-7.0.5-py3-none-any.whl.metadata (5.6 kB)
  Using cached click-8.3.2-py3-none-any.whl.metadata (2.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached pillow-12.2.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (8.8 kB)
  Using cached pydeck-0.9.1-py2.py3-none-any.whl.metadata (4.1 kB)
  Using cached protobuf-7.34.1-cp310-abi3-macosx_10_9_universal2.whl.metadata (595 bytes)
  Using cached pyarrow-23.0.1-cp313-cp313-macosx_12_0_arm64.whl.metadata (3.1 kB)
  Using cached toml-0.10.2-py2.py3-none-any.whl.metadata (7.1 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
  Using cached narwhals-2.19.0-py3-none-any.whl.metadata (14 kB)
  Using cached gitdb-4.0.12-py